---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [ ]:
# Placeholder for your implementation
import zipfile
import helpers
import numpy as np
import litellm 

In [3]:

zip_path = "../data/fordham-website.zip"

# Load data from zipfile
documents = []
with zipfile.ZipFile(zip_path, 'r') as zipf:
    for filename in zipf.namelist():
        if filename.endswith('.md'):
            with zipf.open(filename) as file:
                content = file.read().decode('utf-8')
                documents.append({
                    'filename': filename,
                    'content': content
                })

# Example: print the number of files and first file info
print(f"Loaded {len(documents)} markdown files from the zip archive.")
if documents:
    print("First document:", documents[0]['filename'])

Loaded 9551 markdown files from the zip archive.
First document: index.md


In [4]:
print(documents[12]['content'])

https://www.fordham.edu/summer-session

# Summer Session at Fordham

[Summer classes](/summer-session/summer-courses/course-descriptions-by-subject/)are a great way to get ahead on your college coursework.

- Online synchronous and asynchronous classes are open to Fordham and visiting students!

- May 26 - June 25, 2026 | June 30 - August 4, 2026 | May 26 - August 4, 2026

- Classes available at the Rose Hill Campus in the Bronx and at the Lincoln Center Campus in Manhattan.


---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [5]:
import re

def chunk_markdown(content, chunk_size=900):
    """
    Splits markdown by headers (#, ##, ###) to preserve context,
    then ensures chunks stay within the character limit.
    Ensures chunks do not cut words in the middle (splits at whitespace only).
    If a chunk goes past the limit, move the last sentence to the next chunk.
    """
    header_pattern = r'(^#{1,6}\s+.*$)'
    sections = re.split(header_pattern, content, flags=re.MULTILINE)

    final_chunks = []
    current_chunk = ""

    # Sentence splitter as before
    sentence_splitter = re.compile(r'(?<=[.!?])(?=\s+)')

    for section in sections:
        section = section.strip()
        if not section:
            continue

        # If adding this section exceeds the limit, finalize current chunk and move last sentence to next
        if len(current_chunk) + len(section) > chunk_size:
            if current_chunk:
                # Split current_chunk by sentences, ensuring not to remove the first letter
                sentences = [s.strip() for s in sentence_splitter.split(current_chunk) if s.strip()]
                if len(sentences) > 1:
                    # Keep all but the last sentence in the current chunk
                    chunk_to_add = " ".join(sentences[:-1]).strip()
                    leftover = sentences[-1].strip()
                    if chunk_to_add:
                        final_chunks.append(chunk_to_add)
                    current_chunk = leftover
                else:
                    final_chunks.append(current_chunk)
                    current_chunk = ""

            # Consider the section: if still too big, split by sentence, 
            # but do not break words, only cut at spaces.
            if len(section) > chunk_size:
                sentences = [s.strip() for s in sentence_splitter.split(section) if s.strip()]
                temp_chunk = ""
                for sentence in sentences:
                    if not sentence:
                        continue
                    sentence = sentence.strip()
                    # If the whole sentence won't fit, we break by whitespace only
                    sentence_parts = []
                    if len(sentence) > chunk_size:
                        start = 0
                        while start < len(sentence):
                            # Find the max end within chunk_size that ends at whitespace or end
                            end = min(start + chunk_size, len(sentence))
                            part = sentence[start:end]
                            if end < len(sentence):
                                # Move end back to last whitespace before cut
                                last_space = part.rfind(" ")
                                if last_space > 0:
                                    end = start + last_space
                                    part = sentence[start:end]
                            part = part.rstrip()
                            if part:
                                sentence_parts.append(part)
                            start = end
                    else:
                        sentence_parts = [sentence]
                    # Add the sentence parts ensuring the chunk does not break words
                    for part in sentence_parts:
                        if len(temp_chunk) + len(part) + 1 > chunk_size:
                            if temp_chunk:
                                final_chunks.append(temp_chunk.strip())
                            temp_chunk = part
                        else:
                            temp_chunk = f"{temp_chunk} {part}".strip() if temp_chunk else part
                if temp_chunk:
                    current_chunk = temp_chunk
                else:
                    current_chunk = ""
            else:
                current_chunk = (current_chunk + "\n\n" + section).strip() if current_chunk else section
        else:
            current_chunk = f"{current_chunk}\n\n{section}".strip() if current_chunk else section

    if current_chunk:
        final_chunks.append(current_chunk)

    return final_chunks

# Usage
chunks = chunk_markdown(documents[0]['content'])

In [6]:
print(documents[0]['content'])

https://www.fordham.edu/

## Doing Good That Becomes Greater As The Jesuit University of New York

We’re located in New York City—driven by our Jesuit values and tackling today’s most pressing issues at the center of the world stage.

## We Are Leaders, Dreamers, Achievers, And Doers

With sound hearts, strong minds, and the wisdom to take charge, generations of Rams have found what they have needed to grow—the opportunities, connections, and support of this community.

## From Winding Elms to Bustling City Blocks

With residential campuses in the Bronx and Manhattan, as well as campuses in Westchester and London, Fordham provides endless opportunities to start working toward your career and building the life you want.


## We’re Drawn to Where We’re Needed Most

Explore how our values come to life: how Fordham’s students, faculty, and alumni contribute to society and make lives better.

**Notice of Nondiscriminatory Policy:**

Fordham University admits students of any race, color, nat

In [7]:
chunks

['https://www.fordham.edu/\n\n## Doing Good That Becomes Greater As The Jesuit University of New York\n\nWe’re located in New York City—driven by our Jesuit values and tackling today’s most pressing issues at the center of the world stage. ## We Are Leaders, Dreamers, Achievers, And Doers\n\nWith sound hearts, strong minds, and the wisdom to take charge, generations of Rams have found what they have needed to grow—the opportunities, connections, and support of this community. ## From Winding Elms to Bustling City Blocks\n\nWith residential campuses in the Bronx and Manhattan, as well as campuses in Westchester and London, Fordham provides endless opportunities to start working toward your career and building the life you want.',
 'Explore how our values come to life: how Fordham’s students, faculty, and alumni contribute to society and make lives better. **Notice of Nondiscriminatory Policy:**\n\nFordham University admits students of any race, color, national and ethnic origin to all t

In [8]:
for document in documents:
    document['chunks'] = chunk_markdown(document['content'])

In [9]:
documents[5]['chunks']

["https://www.fordham.edu/undergraduate-financial-aid\n\n# Student Financial Services\n\n## Financial aid can help make a Fordham education possible\n\n![Students relax and unwind on the grass in front of Keating Hall on the Rose Hill campus](https://pxl-fordhamedu.terminalfour.net/prod01/channel_2/media/home/commonly-used-images/campus-beauty/SCostinHardy-20170503-0001.jpg)\n\n## Undergraduate Financial Aid\n\nThis aid comes from Fordham's own resources and is offered to full-time traditional undergraduate students to augment aid from all other sources.",
 '[View undergraduate financial aid](/student-financial-services/undergraduate-financial-aid/)\n\n![Lincoln Center Columbus Circle walking in streets](https://pxl-fordhamedu.terminalfour.net/prod01/channel_2/media/home/commonly-used-images/campus-beauty/KGamble-20140908-0356.jpg)\n\n## Tuition & Payments\n\nWe encourage you to review information on tuition and costs of attendance and to familiarize yourself with the processes of appl

---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [ ]:
# Use OpenAI embeddings (sync version) with error handling, but ensure chunk size is safe for OpenAI's context window
# to avoid "maximum context length" errors.

import openai
import numpy as np
import time
from dotenv import load_dotenv
import os

# Load .env variables (including OPENAI_API_KEY)
load_dotenv()

# Choose your model and define the max token size OpenAI supports for this model.
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 32  # OpenAI API supports up to 2048, but 32-128 is safe for most use cases
MAX_TOKENS_PER_INPUT = 8192  # Per OpenAI - for text-embedding-3-small

import tiktoken
def count_tokens(text, model=OPENAI_EMBEDDING_MODEL):
    """Estimate the number of tokens for a chunk using tiktoken."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

def filter_chunks_by_length(chunks, max_tokens=MAX_TOKENS_PER_INPUT, verbose=True):
    safe_chunks = []
    too_long = []
    for chunk in chunks:
        tokens = count_tokens(chunk)
        if tokens > max_tokens:
            too_long.append((chunk, tokens))
        else:
            safe_chunks.append(chunk)
    if verbose and too_long:
        print(f"{len(too_long)} chunks skipped due to exceeding {max_tokens} tokens:")
        for chunk, tokens in too_long[:3]:
            print(f"  Skipped chunk of {tokens} tokens. Chunk preview:\n{chunk[:256]}...\n")
        if len(too_long) > 3:
            print(f"...({len(too_long)-3} more too-long chunks not shown)")
    return safe_chunks

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def get_openai_embeddings_sync(
    chunks, model=OPENAI_EMBEDDING_MODEL, batch_size=BATCH_SIZE, max_retries=3, verbose=True
):
    """
    Batch OpenAI embedding API calls synchronously for a list of chunks.
    Returns: dict(chunk_text -> np.ndarray)
    Skips over-long chunks.
    """
    safe_chunks = filter_chunks_by_length(chunks, max_tokens=MAX_TOKENS_PER_INPUT, verbose=verbose)
    embeddings = {}
    total = len(safe_chunks)
    for i in range(0, total, batch_size):
        batch = safe_chunks[i:i+batch_size]
        retry = 0
        while retry < max_retries:
            try:
                response = client.embeddings.create(
                    input=batch,
                    model=model,
                )
                for chunk, emb in zip(batch, response.data):
                    embeddings[chunk] = np.array(emb.embedding, dtype=np.float32)
                if verbose:
                    print(f"Embedded batch {i // batch_size + 1} / {((total-1)//batch_size)+1}")
                break
            except Exception as e:
                retry += 1
                if retry == max_retries:
                    print(f"Error on batch {i // batch_size + 1}: {e}. Giving up.")
                    raise
                else:
                    wait_time = 2 ** retry
                    print(f"Retrying batch {i // batch_size + 1} due to error: {e} (attempt {retry})")
                    time.sleep(wait_time)
    return embeddings

# Build list of all chunks (assuming `document['chunks']` is list of str)
all_chunks = []
for document in documents:
    all_chunks.extend(document["chunks"])

# Only embed unique chunks
unique_chunks = list(set(all_chunks))

# ---- Time and test with first 100 unique chunks ----
sample_chunks = unique_chunks[:100]
print(f"Timing OpenAI embeddings for {len(sample_chunks)} chunks (sync)...")
start = time.time()
sample_embs = get_openai_embeddings_sync(sample_chunks)
elapsed = time.time() - start
print(f"Elapsed time for 100 chunks: {elapsed:.2f} seconds")

# ---- Now run on all unique chunks ----
print(f"Embedding all {len(unique_chunks)} unique chunks now (sync)...")
start_all = time.time()
embeddings = get_openai_embeddings_sync(unique_chunks)
elapsed_all = time.time() - start_all
print(f"Elapsed time for ALL ({len(unique_chunks)}) chunks: {elapsed_all:.2f} seconds")

Timing OpenAI embeddings for 100 chunks (sync)...
Embedded batch 1 / 4
Embedded batch 2 / 4
Embedded batch 3 / 4
Embedded batch 4 / 4
Elapsed time for 100 chunks: 3.80 seconds
Embedding all 57185 unique chunks now (sync)...
Embedded batch 1 / 1788
Embedded batch 2 / 1788
Embedded batch 3 / 1788
Embedded batch 4 / 1788
Embedded batch 5 / 1788
Embedded batch 6 / 1788
Embedded batch 7 / 1788
Embedded batch 8 / 1788
Embedded batch 9 / 1788
Embedded batch 10 / 1788
Embedded batch 11 / 1788
Embedded batch 12 / 1788
Embedded batch 13 / 1788
Embedded batch 14 / 1788
Embedded batch 15 / 1788
Embedded batch 16 / 1788
Embedded batch 17 / 1788
Embedded batch 18 / 1788
Embedded batch 19 / 1788
Embedded batch 20 / 1788
Embedded batch 21 / 1788
Embedded batch 22 / 1788
Embedded batch 23 / 1788
Embedded batch 24 / 1788
Embedded batch 25 / 1788
Embedded batch 26 / 1788
Embedded batch 27 / 1788
Embedded batch 28 / 1788
Embedded batch 29 / 1788
Embedded batch 30 / 1788
Embedded batch 31 / 1788
Embedded b

In [10]:
# Use OpenAI embeddings (sync version) with error handling, but ensure chunk size is safe for OpenAI's context window
# to avoid "maximum context length" errors.

import openai
import numpy as np
import time
from dotenv import load_dotenv
import os

# Load .env variables (including OPENAI_API_KEY)
load_dotenv()

# Choose your model and define the max token size OpenAI supports for this model.
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 32  # OpenAI API supports up to 2048, but 32-128 is safe for most use cases
MAX_TOKENS_PER_INPUT = 8192  # Per OpenAI - for text-embedding-3-small

import tiktoken
def count_tokens(text, model=OPENAI_EMBEDDING_MODEL):
    """Estimate the number of tokens for a chunk using tiktoken."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

def filter_chunks_by_length(chunks, max_tokens=MAX_TOKENS_PER_INPUT, verbose=True):
    safe_chunks = []
    too_long = []
    for chunk in chunks:
        tokens = count_tokens(chunk)
        if tokens > max_tokens:
            too_long.append((chunk, tokens))
        else:
            safe_chunks.append(chunk)
    if verbose and too_long:
        print(f"{len(too_long)} chunks skipped due to exceeding {max_tokens} tokens:")
        for chunk, tokens in too_long[:3]:
            print(f"  Skipped chunk of {tokens} tokens. Chunk preview:\n{chunk[:256]}...\n")
        if len(too_long) > 3:
            print(f"...({len(too_long)-3} more too-long chunks not shown)")
    return safe_chunks

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def get_openai_embeddings_sync(
    chunks, model=OPENAI_EMBEDDING_MODEL, batch_size=BATCH_SIZE, max_retries=3, verbose=True
):
    """
    Batch OpenAI embedding API calls synchronously for a list of chunks.
    Returns: dict(chunk_text -> np.ndarray)
    Skips over-long chunks.
    """
    safe_chunks = filter_chunks_by_length(chunks, max_tokens=MAX_TOKENS_PER_INPUT, verbose=verbose)
    embeddings = {}
    total = len(safe_chunks)
    for i in range(0, total, batch_size):
        batch = safe_chunks[i:i+batch_size]
        retry = 0
        while retry < max_retries:
            try:
                response = client.embeddings.create(
                    input=batch,
                    model=model,
                )
                for chunk, emb in zip(batch, response.data):
                    embeddings[chunk] = np.array(emb.embedding, dtype=np.float32)
                if verbose:
                    print(f"Embedded batch {i // batch_size + 1} / {((total-1)//batch_size)+1}")
                break
            except Exception as e:
                retry += 1
                if retry == max_retries:
                    print(f"Error on batch {i // batch_size + 1}: {e}. Giving up.")
                    raise
                else:
                    wait_time = 2 ** retry
                    print(f"Retrying batch {i // batch_size + 1} due to error: {e} (attempt {retry})")
                    time.sleep(wait_time)
    return embeddings

In [ ]:
# Save the computed OpenAI embeddings to disk as a .npy file
import os

def get_available_filename(base_name):
    """
    Returns an available filename by incrementing a number if file already exists.
    E.g., "openai_embs.npy", "openai_embs_1.npy", ...
    """
    if not os.path.exists(base_name):
        return base_name
    fn, ext = os.path.splitext(base_name)
    counter = 1
    while True:
        candidate = f"{fn}_{counter}{ext}"
        if not os.path.exists(candidate):
            return candidate
        counter += 1

embs_filename = get_available_filename('openai_embs.npy')
np.save(embs_filename, embeddings)
print(f"Saved embeddings to {embs_filename}")

Saved embeddings to openai_embs_1.npy


---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

In [11]:
def retrieve_relevant_chunks_semantic(question, embeddings, chunks, k=10):
    """
    Retrieve the most relevant chunks given a question using semantic similarity.

    Args:
        question (str): The user's question.
        embeddings (np.ndarray): Pre-computed embeddings for each chunk, shape (num_chunks, emb_dim).
        chunks (list[str]): List of text chunks.
        k (int, optional): Number of chunks to retrieve. Defaults to 10.

    Returns:
        list[str]: Top-k most relevant chunks.
    """
    # 1. Embed the question using OpenAI embeddings
    query_embs = get_openai_embeddings_sync([question])

    # Ensure the embedding is always a numpy array, not a dict (fixes ufunc TypeError)
    if isinstance(query_embs, dict):
        # Some APIs may return a dict indexed by the input strings
        query_emb = np.array(query_embs[question])
    elif isinstance(query_embs, (list, np.ndarray)):
        # Most APIs return a batch of vectors
        if len(query_embs) == 0:
            raise ValueError("No embedding returned for the question.")
        query_emb = np.array(query_embs[0])
    else:
        raise TypeError("get_openai_embeddings_sync returned unexpected type.")

    # 2. Ensure embeddings is a numpy array and not a dict
    if isinstance(embeddings, dict):
        # Convert dict of embeddings to a 2D array, ordered by chunks
        embeddings = np.array([embeddings[c] for c in chunks])
    elif not isinstance(embeddings, np.ndarray):
        embeddings = np.array(embeddings)

    # 3. Compute similarities (cosine similarity)
    similarities = helpers.batch_cosine_similarity(query_emb, embeddings)

    # 4. Get top-k chunk indices
    top_k_idx = np.argsort(-similarities)[:k]

    # 5. Return the corresponding chunks
    return [chunks[i] for i in top_k_idx]


In [12]:
def retrieve_relevant_chunks_bm25(question, embeddings, chunks, k=10):
    """
    Retrieve the most relevant chunks given a question using BM25.

    Args:
        question (str): The user's question.
        chunks (list[str]): List of text chunks.
        k (int, optional): Number of chunks to retrieve. Defaults to 10.

    Returns:
        list[str]: Top-k most relevant chunks.
    """
    # Use the helpers.bm25 function for scoring chunks

    # Defensive: Ensure chunks is a list/iterable of str, not an int or other type.
    if not isinstance(chunks, (list, tuple)) or (len(chunks) > 0 and not isinstance(chunks[0], str)):
        raise TypeError("chunks must be a list of strings (text chunks). Received type: {}".format(type(chunks)))
    index, doc_lengths = helpers.build_index(chunks)
    scores = helpers.score_bm25(question, index, num_docs=len(chunks), doc_lengths=doc_lengths)

    # Get top-k chunk indices based on BM25 score
    top_k_idx = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]

    return [chunks[i] for i in top_k_idx]


In [13]:
def retrieve_relevant_chunks_hybrid(question, embeddings, chunks, k=10, alpha=0.5):
    """
    Combines BM25 and Semantic search results using a weighted alpha.
    """
    # 1. Get Semantic Scores (Cosine Similarity)
    # Re-using your semantic logic: returns chunks, but we need scores for the formula
    query_embs = get_openai_embeddings_sync([question])
    query_emb = np.array(query_embs[question])
    
    # Convert dict of embeddings to 2D array ordered by chunks list
    chunk_embeddings = np.array([embeddings[c] for c in chunks])
    semantic_scores = helpers.batch_cosine_similarity(query_emb, chunk_embeddings)

    # 2. Get BM25 Scores
    # Build BM25 inverted index over chunks for ad-hoc scoring
    index, doc_lengths = helpers.build_index(chunks)
    bm25_raw_scores = helpers.score_bm25(question, index, num_docs=len(chunks), doc_lengths=doc_lengths)

    # 3. Normalize BM25 scores to [0, 1] range
    bm25_min = bm25_raw_scores.min()
    bm25_max = bm25_raw_scores.max()
    
    if bm25_max - bm25_min > 0:
        bm25_normalized = (bm25_raw_scores - bm25_min) / (bm25_max - bm25_min)
    else:
        bm25_normalized = bm25_raw_scores

    # 4. Combine scores using the Convex Combination formula
    hybrid_scores = (alpha * bm25_normalized) + ((1 - alpha) * semantic_scores)

    # 5. Get top-k indices and return chunks
    top_k_idx = np.argsort(-hybrid_scores)[:k]
    
    return [chunks[i] for i in top_k_idx]

---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [14]:
import openai

def generate_answer(question, retrieved_chunks, model="gpt-4o-mini", temperature=0.2, max_tokens=512):
    """
    Generate an answer using an LLM, given a question and retrieved context chunks.

    Args:
        question (str): The user's question.
        retrieved_chunks (list[str]): Relevant text chunks retrieved via semantic search.
        model (str): OpenAI model to use.
        temperature (float): Sampling temperature.
        max_tokens (int): Maximum tokens for the response.

    Returns:
        str: The generated answer from the LLM.
    """
    # 1. Build prompt
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""You are an assistant helping to answer questions about Fordham University. 

Use the context provided below to answer the user's question as accurately as possible. If the context does not contain the answer, say 'I could not find the answer in the provided information.' Do not make up answers or include information not in the context.

Context:
---------
{context}
---------

Question: {question}

Answer:"""
    
    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": "You answer questions based only on the provided context. If the answer is not present, say you could not find it."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response['choices'][0]['message']['content'].strip()

---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [ ]:
# Allow the user to input a question, then answer via the generate_answer function
# Restructure all the chunks from the map documents: aggregate all values in the "chunks" key into a list

chunks = []
for doc in documents:
    if "chunks" in doc and isinstance(doc["chunks"], list):
        chunks.extend(doc["chunks"])


# Create a list of user questions to test
user_questions = [
    "What is the application deadline for undergraduates at Fordham University?",
    "Does Fordham University offer financial aid for international students?",
 #   "Where are the campuses of Fordham University located?",
  #  "How can I schedule a campus visit to Fordham?",
  #  "What are the core values or mission of Fordham University?",
  #  "What are the concentrations Fordham MBA offers?",
  #  "What are the different types of Masters Programs that Fordham offers?",
  #  'Tell me about the Executive MBA program.'

]

# Prepare a list to hold the results
qa_results = []
embs = None
# Load from file if no embeddings in varialbe
if not embeddings:
    embs = np.load('openai_embs_1.npy', allow_pickle=True).item()

else:
    embs = embeddings

# For each question, retrieve chunks and generate an answer, then record all
for user_question in user_questions:
    retrieved_chunks = retrieve_relevant_chunks_hybrid(user_question, embs, chunks, k=5)
    answer = generate_answer(user_question, retrieved_chunks)
    # Store the question, top retrieved chunks, and answer in a map
    qa_results.append({
        "question": user_question,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer,
    })

# Optionally, print results for inspection
for idx, qa in enumerate(qa_results):
    print(f"\n--- QA {idx+1} ---")
    print("Question:", qa["question"])
    print("\nRetrieved Chunks:")
    for chunk in qa["retrieved_chunks"]:
        print("-", chunk[:250], "...")  # print first 250 chars for brevity
    print("\nAnswer:")
    print(qa["answer"])


Embedded batch 1 / 1
Embedded batch 1 / 1

--- QA 1 ---
Question: What is the application deadline for undergraduates at Fordham University?

Retrieved Chunks:
- Students seeking short-term, full-time study at Fordham for a fall or spring semester while matriculated at another college or university should apply for visiting student admission. To be considered, please complete the online [Visiting Student Appl ...
- - Begin Fordham undergraduate studies the semester after you successfully complete Via Fordham! All documents are due by the application deadline. - Apply for Via Fordham before the application deadline. ...
- Please review the deadlines, details, and

[instructions for international transfer students](/undergraduate-admission/international-students/international-transfer-students/). -
[Common Application](https://apply.commonapp.org/login?ma=98&tref=3003) ...
- *To be considered for institutional aid (including early action applicants accepted during the regular decision pr

---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

## Evaluation Steps 
- This section contains the steps and code for evaluating your retrieval-augmented generation (RAG) system.
- The following cells introduce evaluation methods, provide code for measuring retrieval quality, 
- and discuss ways to experiment and improve the performance of your system.
- Because it is imfeasible to get the relevance score for each document for every query, we reach the 50 most relevant chunks for each query and grade them for relevance and precision calculations at the end

In [26]:
def llm_relevance_grade(question, chunk, gold_answer, model="gpt-4.0-mini", temperature=0.0):
    """
    Uses an LLM to assign a relevance grade to the retrieved chunk for a given (question, chunk, gold_answer).
    Grade: 2 = very relevant, 1 = somewhat relevant, 0 = not relevant.
    """
    prompt = f"""
You are an assistant evaluating retrieval results for a question-answer task.

- User Question: {question}
- Gold/Ideal Answer: {gold_answer}
- Retrieved Chunk (to assess): {chunk}

Determine how relevant the retrieved chunk is for answering the user's question, using the following scale:

2 = Very relevant (essential details, directly supports or answers the question)
1 = Somewhat relevant (some overlap, partially useful, but missing key info or less direct)
0 = Not relevant (off-topic, does not help answer the question)

Only return a single integer: 2, 1, or 0, and nothing else.
"""
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=1,
    )
    try:
        grade = int(response['choices'][0]['message']['content'].strip())
        return grade if grade in [0, 1, 2] else 0
    except Exception:
        return 0

def ndcg_relevance_for_query(question, retrieved_chunks, gold_answer, k=5, model="gpt-3.5-turbo"):
    """
    Grades each retrieved chunk for a query using an LLM, computes NDCG@k.

    Returns (ndcg, grades list).
    """
    grades = []
    for chunk in retrieved_chunks:
        grade = llm_relevance_grade(question, chunk, gold_answer, model=model)
        grades.append(grade)
    # Correction: NDCG@k must use only the top-k grades, not all retrieved.
    ndcg = helpers.calculate_ndcg(grades[:k], k)
    return ndcg, grades

def generate_general_fordham_questions(n=10, model="gpt-3.5-turbo"):
    """
    Uses the LLM to generate a list of very general questions about Fordham University and its programs.
    Returns a list of questions.
    """
    prompt = (
        "Generate a numbered list of {} very general questions about Fordham University and the programs it offers. "
        "The questions should be broad (not specific to individuals), cover topics such as admissions, campus life, academic programs, faculty, research, financial aid, and career outcomes. Each question should be unique and relevant to prospective or current students.\n\n"
        "Numbered List:\n".format(n)
    )
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=256,
    )
    # Parse response into a list of question strings
    if isinstance(response, dict):
        content = response['choices'][0]['message']['content']
    else:
        content = getattr(response.choices[0].message, "content", None) or response.content
    questions = []
    for line in content.strip().split('\n'):
        # Extract question after the number
        if "." in line:
            q = line.split(".", 1)[1].strip()
            if q: questions.append(q)
    return questions

def recall_at_k_for_query(question, retrieved_chunks, gold_answer, k=5):
    """
    Estimate recall@k by comparing RAG answer to the retrieved chunks that contributed to it.
    We use LLM grades as proxies for relevance at each position.
    Returns (recall@k, grades_at_k).
    """
    # Use only the top-k chunks and grade them
    grades_at_k = []
    for chunk in retrieved_chunks[:k]:
        grade = llm_relevance_grade(question, chunk, gold_answer, model="gpt-3.5-turbo")
        grades_at_k.append(grade)
    relevant_in_k = sum(1 for g in grades_at_k if g > 0)
    # Now, to get "total relevant in all retrieved": grade all retrieved chunks!

    all_grades = []
    for chunk in retrieved_chunks:
        grade = llm_relevance_grade(question, chunk, gold_answer, model="gpt-3.5-turbo")
        all_grades.append(grade)
    n_relevant_total = sum(1 for g in all_grades if g > 0)
    # Prevent dividing by zero if no relevant found in all chunks
    recall = relevant_in_k / n_relevant_total if n_relevant_total > 0 else 0.0
    return recall, grades_at_k


In [24]:
# Generate general Fordham questions
general_questions = generate_general_fordham_questions(n=10)

In [27]:

def evaluate_questions(questions, chunk_retrieval_fn, answer_fn=None, k=5):
    """
    Evaluate a list of questions using the given chunk retrieval function and answer generation function.
    Returns a list of dictionaries with QA, retrieval, NDCG, and recall@10 results.
    
    Args:
        questions (list of str): Questions to evaluate.
        chunk_retrieval_fn (callable): Function that takes (question, k) and returns the relevant chunks.
        answer_fn (callable, optional): Function that takes (question, chunks) and returns an answer. Defaults to generate_answer.
        k (int, optional): Number of top chunks to retrieve. Defaults to 5.
    
    Returns:
        list of dict: Results for each question.
    """
    if answer_fn is None:
        answer_fn = generate_answer
    results = []
    print("\nGeneral Fordham Questions Generated by LLM:\n")
    for q in questions:
        print("-", q)
        retrieved_chunks = chunk_retrieval_fn(q, embs, chunks, k = 100)
        answer = answer_fn(q, retrieved_chunks)
        result = {
            "question": q,
            "retrieved_chunks": retrieved_chunks,
            "answer": answer
        }

        # Compute NDCG metric and LLM relevance grades for each retrieved chunk
        result['ndcg'], result['grades'] = ndcg_relevance_for_query(q, retrieved_chunks, answer)

        top_k_grades = result.get('grades', [])[:k]  # Ensure grades matches k

        # Calculate Precision@k: proportion of relevant items in the top-k (numerator cannot exceed k)
        relevant_in_top_k = sum(1 for g in top_k_grades if g > 0)
        result[f'precision@{k}'] = relevant_in_top_k / k

        # Calculate Recall@k: proportion of relevant items in top-k relative to all relevant in retrieved_chunks
        # total_possible_relevant is all chunks the judge deemed relevant in full result
        total_possible_relevant = sum(1 for g in result.get('grades', []) if g > 0)
        result[f'recall@{k}'] = relevant_in_top_k / total_possible_relevant if total_possible_relevant > 0 else 0

        # Optionally keep the grades for top k
        result[f'grades@{k}'] = top_k_grades

        results.append(result)

    # Display results
    print(f"\nNDCG Results for Generated General Fordham Questions for {chunk_retrieval_fn}:\n")
    for i, (q, result) in enumerate(zip(questions, results)):
        print(f"Q{i+1}: {q}")
        print(f"  NDCG: {result.get('ndcg', 'N/A')}")
        print(f"  Grades: {result.get('grades', 'N/A')}")
        print(f"  Recall@{k}: {result.get(f'recall@{k}', 'N/A')}")
        print(f"  Precision@{k}: {result.get(f'precision@{k}', 'N/A')}")

    # Print average NDCG and Recall@10 across all questions
    avg_ndcg = sum([result.get('ndcg', 0.0) for result in results]) / len(results) if results else 0.0
    avg_recall = sum([result.get(f'recall@{k}', 0.0) for result in results]) / len(results) if results else 0.0
    print(f"\nAverage NDCG: {avg_ndcg:.4f}")
    print(f"Average Recall@{k}: {avg_recall:.4f}\n")

    return results

import pandas as pd

result_list = []
result_list.append(evaluate_questions(general_questions, retrieve_relevant_chunks_bm25))
result_list.append(evaluate_questions(general_questions, retrieve_relevant_chunks_hybrid))
result_list.append(evaluate_questions(general_questions, retrieve_relevant_chunks_semantic))



General Fordham Questions Generated by LLM:

- What are the admission requirements for undergraduate and graduate programs at Fordham University?
- How diverse is the student body at Fordham University and what resources are available for underrepresented groups?
- What types of academic programs and majors are offered at Fordham University?
- Can students participate in research opportunities with faculty members at Fordham University?
- How does Fordham University support students in finding internships and career opportunities?
- What is the average class size at Fordham University and how accessible are professors to students?
- What campus facilities and resources are available to students, such as libraries, labs, and study spaces?
- How does Fordham University support students in terms of financial aid and scholarships?
- What extracurricular activities and clubs are available for students to participate in at Fordham University?
- What are some examples of successful career ou

In [28]:
# Collect averages including precision
average_ndcgs = []
average_recalls = []
average_precisions = []
method_names = ["BM25", "Hybrid", "Semantic"]

for results, method in zip(result_list, method_names):
    avg_ndcg = sum([result.get('ndcg', 0.0) for result in results]) / len(results) if results else 0.0
    k = 5  # or 10 depending on the value used in evaluate_questions
    avg_recall = sum([result.get(f'recall@{k}', 0.0) for result in results]) / len(results) if results else 0.0
    avg_precision = sum([result.get(f'precision@{k}', 0.0) for result in results]) / len(results) if results else 0.0
    average_ndcgs.append(avg_ndcg)
    average_recalls.append(avg_recall)
    average_precisions.append(avg_precision)

summary_df = pd.DataFrame({
    "Method": method_names,
    "Average NDCG": average_ndcgs,
    "Average Recall": average_recalls,
    "Average Precision": average_precisions
})

display(summary_df)


,Method,Average NDCG,Average Recall,Average Precision
0,BM25,0.851790,0.076228,0.60
1,Hybrid,0.929073,0.086053,0.80
2,Semantic,0.946578,0.083747,0.88


### Evaluation Summary

| Method   | Average NDCG | Average Recall | Average Precision |
|----------|--------------|---------------|------------------|
| BM25     | 0.851790     | 0.076228      | 0.60             |
| Hybrid   | 0.929073     | 0.086053      | 0.80             |
| Semantic | 0.946578     | 0.083747      | 0.88             |

**Analysis:**

- **Semantic search** performed the best overall, showing the highest Average NDCG (0.946578) and Precision (0.88), with Recall slightly below Hybrid but greater than BM25.
- **Hybrid search** yields better performance than BM25, with substantial improvement in both NDCG and Precision, and the highest Recall (0.086053).
- **BM25** lags behind both other methods in all metrics, though it provides a reasonable baseline.

**Interpretation:**
- If you care most about the *quality* and *relevance* of the top results (NDCG/Precision), Semantic is clearly superior.
- For *coverage* (Recall), Hybrid performs best but the difference over Semantic is very small.
- Precision rises steadily from BM25 to Hybrid to Semantic, indicating that more advanced retrieval methods are surfacing more correct/relevant results in the top positions.

**Conclusion:**  
*Upgrading from BM25 to a Hybrid or especially a Semantic retriever can yield substantial improvements in both ranking quality and precision in a RAG system.*

### Save Chunks to JSON for App

In [9]:
import json
chunks_1 = []
for doc in documents:
    chunks_1.extend(doc['chunks'])
with open("chunks_1.json", "w", encoding="utf-8") as f:
    json.dump(chunks_1, f, ensure_ascii=False, indent=2)

---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.